In [1]:

import numpy as np

from tslearn.datasets import UCR_UEA_datasets
import torch
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
import pandas as pd
from sklearn.linear_model import LogisticRegression
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from uni2ts.model.moirai import MoiraiModule
import torch.nn as nn
import torch.optim as optim
from sklearn.linear_model import RidgeClassifierCV
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
#import seaborn as sns

from encoder import MoiraiEncoder

/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Utils

In [2]:
PATCH_SIZES = [8, 16, 32, 64]

SIZE = "small"
DEVICE = "cuda"
NUM_VARS = 6

DO_MASK = True
KEEP_MASK_EMBEDDING = False

In [3]:
def preprocess_data(
    data: np.ndarray,
    *,
    device: str | torch.device = "cpu",
    dtype: torch.dtype = torch.float32,
):
    """
    data: np.ndarray of shape (N, T, V) = (n_individual, time, variate)
    Assumes NO missing values and NO padding in the raw data.
    """

    if not isinstance(data, np.ndarray):
        raise TypeError("data must be a numpy.ndarray")
    if data.ndim != 3:
        raise ValueError(f"Expected shape (N,T,V), got {data.shape}")

    N, T, V = data.shape

    # (N,T,V)
    past_target = torch.as_tensor(data, dtype=dtype, device=device)

    # observed mask: (N,T,V) all True no missing values
    past_observed_target = torch.ones((N, T, V), dtype=torch.bool, device=device)

    # padding mask: (N,T) all if no padding
    past_is_pad = torch.zeros((N, T), dtype=torch.bool, device=device)

    return past_target, past_observed_target, past_is_pad

In [4]:
def apply_pooling_pt(Z_tensor, method, num_vars=NUM_VARS):
    N, S, F = Z_tensor.shape
    P = S // num_vars # Calcul automatique du nombre de patches par variable
    
    # On reshape le tenseur pour séparer les Variables et les Patches
    # Forme résultante : (Batch, Variables, Patches, Features)
    Z_reshaped = Z_tensor.view(N, num_vars, P, F)
    
    # Basique et Global
    if method == "flatten":
        return Z_tensor.reshape(N, -1)
        
    elif method == "global_mean":
        return Z_tensor.mean(dim=1)
        
    elif method == "global_max":
        return Z_tensor.max(dim=1).values
        
    elif method == "global_min":
        return Z_tensor.min(dim=1).values
    
    elif method == "global_mean_max_min":
        return torch.cat([
            Z_tensor.mean(dim=1),
            Z_tensor.max(dim=1).values,
            Z_tensor.min(dim=1).values
        ], dim=1)

    # Pooling sur les Patches (on garde les variables distinctes) ---
    # Réduction sur la dimension 2 (Patches). Résultat : (N, num_vars, F), puis on aplatit
    elif method == "mean_over_patches":
        return Z_reshaped.mean(dim=2).reshape(N, -1)
        
    elif method == "max_over_patches":
        return Z_reshaped.max(dim=2).values.reshape(N, -1)
        
    elif method == "min_over_patches":
        return Z_reshaped.min(dim=2).values.reshape(N, -1)
        
    elif method == "mean_max_min_over_patches":
        p_mean = Z_reshaped.mean(dim=2).reshape(N, -1)
        p_max  = Z_reshaped.max(dim=2).values.reshape(N, -1)
        p_min  = Z_reshaped.min(dim=2).values.reshape(N, -1)
        return torch.cat([p_mean, p_max, p_min], dim=1)

    # Pooling sur les Variables (on synchronise les patches entre variables) ---
    # Réduction sur la dimension 1 (Variables). Résultat : (N, P, F), puis on aplatit
    elif method == "mean_over_variables":
        return Z_reshaped.mean(dim=1).reshape(N, -1)
        
    elif method == "max_over_variables":
        return Z_reshaped.max(dim=1).values.reshape(N, -1)
        
    elif method == "min_over_variables":
        return Z_reshaped.min(dim=1).values.reshape(N, -1)
        
    elif method == "mean_max_min_over_variables":
        v_mean = Z_reshaped.mean(dim=1).reshape(N, -1)
        v_max  = Z_reshaped.max(dim=1).values.reshape(N, -1)
        v_min  = Z_reshaped.min(dim=1).values.reshape(N, -1)
        return torch.cat([v_mean, v_max, v_min], dim=1)

    else:
        raise ValueError(f"Method {method} unknow")

pooling_methods = [
    "flatten", 
    "global_mean", "global_max", "global_min", "global_mean_max_min",
    "mean_over_patches", "max_over_patches", "min_over_patches", "mean_max_min_over_patches",
    "mean_over_variables", "max_over_variables", "min_over_variables", "mean_max_min_over_variables"
]

In [5]:
alphas_to_test = [0.1,1,10,30,100,300,1000,10000]

models = {
    'Ridge': RidgeClassifierCV(alphas=alphas_to_test, cv=3),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
}

# Data

Import

In [6]:
ds = UCR_UEA_datasets()
X_train, y_train, X_test, y_test = ds.load_dataset("LSST")

n_classes = len(set(y_train))
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

y_train_tensor = torch.tensor(y_train, dtype=torch.long, device=DEVICE)
y_test_tensor = torch.tensor(y_test, dtype=torch.long, device=DEVICE)

NUM_CLASSES = len(torch.unique(y_train_tensor))

Prepare data

In [7]:
X_train_target_tensor, X_train_is_target_observed, X_train_is_target_padded = (
    preprocess_data(X_train, device=DEVICE)
)

X_test_target_tensor, X_test_is_target_observed, X_test_is_target_padded = (
    preprocess_data(X_test, device=DEVICE)
)
y_train_tensor = torch.tensor(y_train, dtype=torch.long, device=DEVICE)
y_test_tensor = torch.tensor(y_test, dtype=torch.long, device=DEVICE)

num_classes = len(torch.unique(y_train_tensor))
NUM_VARS = 6

print(X_train_target_tensor.shape)
print(X_train_is_target_observed.shape)
print(X_train_is_target_padded.shape)

torch.Size([2459, 36, 6])
torch.Size([2459, 36, 6])
torch.Size([2459, 36])


Creat Z_train and Z_test for each patch size

In [8]:
Z_train_patch = {}
Z_test_patch = {}
for patch in PATCH_SIZES:
    
    # 1. Définir les paramètres et l'encodeur pour la taille de patch actuelle
    MODEL_PARAMS = {
        "patch_size": patch,
        "num_samples": 100,
        "target_dim": 6,
        "feat_dynamic_real_dim": 0,
        "past_feat_dynamic_real_dim": 0,
    }
    if DO_MASK == True:
        prediction_length = patch
    else:
        prediction_length = 0
    encoder = MoiraiEncoder(
        module=MoiraiModule.from_pretrained(f"Salesforce/moirai-1.1-R-{SIZE}"),
        prediction_length=prediction_length,
        context_length=36,
        **MODEL_PARAMS,
    )
    encoder.eval()
    encoder.to(DEVICE)
    
    # 2. Extraire les embeddings (Z_train et Z_test)
    with torch.no_grad():
        Z_train = encoder(
            past_target=X_train_target_tensor,
            past_observed_target=X_train_is_target_observed,
            past_is_pad=X_train_is_target_padded,
        )
        Z_test = encoder(
            past_target=X_test_target_tensor,
            past_observed_target=X_test_is_target_observed,
            past_is_pad=X_test_is_target_padded,
        )
        
    if DO_MASK and not KEEP_MASK_EMBEDDING:
        N, S, F = Z_train.shape
        P = S // NUM_VARS  # Nombre de patchs par variable (Contexte + 1 Masque)
        
        # On reshape pour séparer les variables et les patchs : (Batch, 6 vars, Patchs, Features)
        Z_train_reshaped = Z_train.view(N, NUM_VARS, P, F)
        Z_test_reshaped = Z_test.view(Z_test.shape[0], NUM_VARS, P, F) # Attention à bien récupérer N_test = Z_test.shape[0]
        
        # On slice pour enlever le dernier patch (:-1) sur la dimension des patchs (dim=2)
        Z_train_no_mask = Z_train_reshaped[:, :, :-1, :]
        Z_test_no_mask = Z_test_reshaped[:, :, :-1, :]
        
        # On remet sous la forme attendue (Batch, Séquence_sans_masques, Features)
        Z_train = Z_train_no_mask.reshape(N, -1, F)
        Z_test = Z_test_no_mask.reshape(Z_test.shape[0], -1, F)



    Z_train_patch[patch] = Z_train
    Z_test_patch[patch] = Z_test

In [9]:

print(f"{'Patch Size':<12} | {'Train Shape':<25} | {'Test Shape':<25}")
for patch in PATCH_SIZES:
    train_tensor = Z_train_patch[patch]
    test_tensor = Z_test_patch[patch]
    print(f"{patch:<12} | {str(list(train_tensor.shape)):<25} | {str(list(test_tensor.shape)):<25}")

Patch Size   | Train Shape               | Test Shape               
8            | [2459, 30, 384]           | [2466, 30, 384]          
16           | [2459, 18, 384]           | [2466, 18, 384]          
32           | [2459, 12, 384]           | [2466, 12, 384]          
64           | [2459, 6, 384]            | [2466, 6, 384]           


# Basic pooling with logistic regression and random forest

## Single patch size pooling

We started by applying simple pooling techniques (max, min, or mean) and comparing their performance across various patch sizes. Specifically, we evaluated whether to apply this pooling globally across all patches, locally within a single series, or across corresponding patches from all series.

### Training

We creat a dataframe to stock the results and we fill it

In [10]:
results_dfs = {
    model_name: pd.DataFrame(index=pooling_methods, columns=PATCH_SIZES)
    for model_name in models.keys()
}
for df in results_dfs.values():
    df.index.name = "Pooling \\ Patch"

We train

In [11]:
for patch in PATCH_SIZES:
    Z_train = Z_train_patch[patch].to(DEVICE)
    Z_test = Z_test_patch[patch].to(DEVICE)
    
    for pool in pooling_methods:
        X_train_pool = apply_pooling_pt(Z_train, pool).cpu().numpy()
        X_test_pool = apply_pooling_pt(Z_test, pool).cpu().numpy()

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_pool)
        X_test_scaled = scaler.transform(X_test_pool)
        
        for model_name, clf in models.items():
            clf.fit(X_train_scaled, y_train)
            
            y_pred = clf.predict(X_test_scaled)
            acc = accuracy_score(y_test, y_pred)
            
            results_dfs[model_name].loc[pool, patch] = round(acc, 3)


/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=3.59159e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=3.567e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=3.59439e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditione

### Results

In [12]:
for model_name, df in results_dfs.items():
    print(f"RÉSULTATS POUR : {model_name.upper()}")
    display(df.astype(float))
    
    best_acc = df.max().max()
    best_pool, best_patch = df.astype(float).stack().idxmax()
    print(f"Meilleure combinaison = Patch: {best_patch} | Pooling: '{best_pool}' | Acc: {best_acc:.3f}\n")

RÉSULTATS POUR : RIDGE


,8,16,32,64
Pooling \ Patch,,,,
flatten,0.600,0.624,0.607,0.618
global_mean,0.574,0.592,0.579,0.587
global_max,0.543,0.553,0.532,0.562
global_min,0.545,0.546,0.532,0.557
global_mean_max_min,0.569,0.575,0.554,0.581
mean_over_patches,0.604,0.616,0.603,0.618
max_over_patches,0.589,0.609,0.595,0.618
min_over_patches,0.591,0.599,0.599,0.618
mean_max_min_over_patches,0.607,0.632,0.612,0.623


Meilleure combinaison = Patch: 16 | Pooling: 'mean_max_min_over_patches' | Acc: 0.632

RÉSULTATS POUR : RANDOMFOREST


,8,16,32,64
Pooling \ Patch,,,,
flatten,0.568,0.553,0.539,0.597
global_mean,0.569,0.581,0.551,0.568
global_max,0.531,0.506,0.502,0.558
global_min,0.520,0.525,0.508,0.564
global_mean_max_min,0.566,0.558,0.535,0.570
mean_over_patches,0.584,0.568,0.559,0.597
max_over_patches,0.561,0.558,0.554,0.597
min_over_patches,0.561,0.555,0.554,0.597
mean_max_min_over_patches,0.583,0.571,0.556,0.594


Meilleure combinaison = Patch: 64 | Pooling: 'flatten' | Acc: 0.597



df_ridge_plot = df_results_ridge.astype(float)
df_rf_plot = df_results_rf.astype(float)


fig, axes = plt.subplots(1, 2, figsize=(20, 10), sharey=True)

cmap = "YlGnBu"

sns.heatmap(df_ridge_plot, 
            annot=True,
            fmt=".3f",
            cmap=cmap,
            linewidths=0.5,
            ax=axes[0],
            cbar_kws={'label': 'Accuracy'})
axes[0].set_title("Single Patch Pooling - Accuracy (Ridge)", fontsize=16, fontweight='bold', pad=15)
axes[0].set_xlabel("Patch Size", fontsize=14)
axes[0].set_ylabel("Pooling Method", fontsize=14)
axes[0].tick_params(axis='both', which='major', labelsize=12)

sns.heatmap(df_rf_plot, 
            annot=True, 
            fmt=".3f", 
            cmap=cmap, 
            linewidths=0.5, 
            ax=axes[1], 
            cbar_kws={'label': 'Accuracy'})
axes[1].set_title("Single Patch Pooling - Accuracy (Random Forest)", fontsize=16, fontweight='bold', pad=15)
axes[1].set_xlabel("Patch Size", fontsize=14)
axes[1].tick_params(axis='both', which='major', labelsize=12)

plt.tight_layout()

plt.show()

Pooling over patches rather than series produces the highest accuracy.

## Multi Patch pooling

### Training

In [ ]:
df_multiscale_results = pd.DataFrame(index=pooling_methods, columns=models.keys())
df_multiscale_results.index.name = "Pooling Method"

for pool in pooling_methods:
    X_train_multi = []
    X_test_multi = []
    
    for patch in PATCH_SIZES:
        Z_train = Z_train_patch[patch].to(DEVICE)
        Z_test = Z_test_patch[patch].to(DEVICE)
        
        X_train_pool = apply_pooling_pt(Z_train, pool).cpu().numpy()
        X_test_pool = apply_pooling_pt(Z_test, pool).cpu().numpy()
        
        X_train_multi.append(X_train_pool)
        X_test_multi.append(X_test_pool)
        
    X_train_combined = np.concatenate(X_train_multi, axis=1)
    X_test_combined = np.concatenate(X_test_multi, axis=1)
    
    scaler = StandardScaler()
    X_train_combined_scaled = scaler.fit_transform(X_train_combined)
    X_test_combined_scaled = scaler.transform(X_test_combined)

    for model_name, clf in models.items():
        clf.fit(X_train_combined_scaled, y_train)
        
        y_pred = clf.predict(X_test_combined_scaled)
        acc = accuracy_score(y_test, y_pred)
        
        df_multiscale_results.loc[pool, model_name] = round(acc, 3)

/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=1.92258e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=1.96062e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=1.9049e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: Ill-condition

### Results

In [ ]:
display(df_multiscale_results.astype(float))

best_acc = df_multiscale_results.max().max()
best_pool = df_multiscale_results.max(axis=1).idxmax()
best_model = df_multiscale_results.max(axis=0).idxmax()

print("MEILLEURE COMBINAISON MULTI-ÉCHELLE :")
print(f"Modèle : '{best_model}' | Pooling : '{best_pool}'")
print(f"Précision maximale : {best_acc:.3f}")

,Ridge,RandomForest
Pooling Method,,
flatten,0.646,0.559
global_mean,0.622,0.589
global_max,0.592,0.569
global_min,0.588,0.562
global_mean_max_min,0.618,0.583
mean_over_patches,0.644,0.592
max_over_patches,0.648,0.570
min_over_patches,0.647,0.577
mean_max_min_over_patches,0.655,0.589


MEILLEURE COMBINAISON MULTI-ÉCHELLE :
Modèle : 'Ridge' | Pooling : 'mean_max_min_over_patches'
Précision maximale : 0.655


## Conclusion

pool over patch -> good
16 -> (max, mean, min) > flatten -> position not importante
multi patch -> good

# Advanced Pooling

## Attention Pooling

In [ ]:
class DynamicAttentionClassifier(nn.Module):
    def __init__(self, num_vars, num_classes, patches_counts, in_features=384, mode="global"):
        """
        patches_counts : liste du nombre de patchs par échelle (ex: [30, 18, 12, 6])
        mode : "global" (1), "per_scale" (4), "per_variable" (6), ou "per_both" (24)
        """
        super().__init__()
        self.num_vars = num_vars
        self.patches_counts = patches_counts
        self.num_scales = len(patches_counts)
        self.mode = mode
        
        if mode == "global":
            self.context = nn.Parameter(torch.randn(in_features) * 0.1)
            
        elif mode == "per_scale":
            self.context = nn.Parameter(torch.randn(self.num_scales, in_features) * 0.1)
            
        elif mode == "per_variable":
            self.context = nn.Parameter(torch.randn(self.num_vars, in_features) * 0.1)
            
        elif mode == "per_both":
            self.context = nn.Parameter(torch.randn(self.num_scales, self.num_vars, in_features) * 0.1)
            
            
        self.dropout = nn.Dropout(0.1)
        self.linear = nn.LazyLinear(num_classes)
        
    def forward(self, x):
        B = x.shape[0]
        F = x.shape[-1]
        
        split_sizes = [p * self.num_vars for p in self.patches_counts]
        x_splits = torch.split(x, split_sizes, dim=1)
        
        reshaped_splits = []
        for i, p in enumerate(self.patches_counts):
            reshaped_splits.append(x_splits[i].view(B, self.num_vars, p, F))
            
        if self.mode == "global":
            scores = torch.matmul(x, self.context)
            weights = torch.softmax(scores, dim=1).unsqueeze(-1)
            pooled = (weights * x).sum(dim=1)
            
        elif self.mode == "per_scale":
            pooled_list = []
            for i, xi in enumerate(reshaped_splits):
                xi_flat = xi.view(B, -1, F)
                scores = torch.matmul(xi_flat, self.context[i])
                weights = torch.softmax(scores, dim=1).unsqueeze(-1)
                pooled_list.append((weights * xi_flat).sum(dim=1))
            
            pooled = torch.cat(pooled_list, dim=1)
            
        elif self.mode == "per_variable":
            x_vars = torch.cat(reshaped_splits, dim=2)
            
            scores = (x_vars * self.context.view(1, self.num_vars, 1, F)).sum(dim=-1)
            weights = torch.softmax(scores, dim=2).unsqueeze(-1)
            pooled_vars = (weights * x_vars).sum(dim=2)
            pooled = pooled_vars.view(B, -1)
            
        elif self.mode == "per_both":
            pooled_list = []
            for i, xi in enumerate(reshaped_splits):
                scores = (xi * self.context[i].view(1, self.num_vars, 1, F)).sum(dim=-1)
                weights = torch.softmax(scores, dim=2).unsqueeze(-1)
                pooled_vars = (weights * xi).sum(dim=2)
                pooled_list.append(pooled_vars)
                
            pooled_all = torch.stack(pooled_list, dim=1)
            pooled = pooled_all.view(B, -1)
            
        return self.linear(self.dropout(pooled))

In [ ]:
patch_sizes = [8, 16, 32, 64]
NUM_VARS = 6

patches_counts = [Z_train_patch[p].shape[1] // NUM_VARS for p in patch_sizes]
print(f"Nomber of patch for each scale : {patches_counts}")

Z_train_concat = torch.cat([Z_train_patch[p].to(device) for p in patch_sizes], dim=1)
Z_test_concat  = torch.cat([Z_test_patch[p].to(device) for p in patch_sizes], dim=1)

print(f"Shape Z_train: {Z_train_concat.shape}")

Nomber of patch for each scale : [5, 3, 2, 1]


NameError: name 'device' is not defined

In [ ]:
BATCH_SIZE = 64
EPOCHS = 1000
LEARNING_RATE = 0.0001
EVAL_EVERY_N_EPOCHS = 10
PRINT_LOSS_EVERY = 5 
attention_modes = ['global', 'per_scale', 'per_variable', 'per_both']
df_results = pd.DataFrame(index=attention_modes, columns=["Test Accuracy"])
df_results.index.name = "Attention mode"

train_dataset = TensorDataset(Z_train_concat, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

mode = "global"

model = DynamicAttentionClassifier(
    num_vars=NUM_VARS, 
    num_classes=num_classes, 
    patches_counts=patches_counts, 
    mode=mode
).to(device)

_ = model(Z_train_concat[:2]) # Dummy pass for LazyLinear

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    
    for batch_z, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_z)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * batch_z.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    

    if (epoch + 1) == 1 or (epoch + 1) % PRINT_LOSS_EVERY == 0:
        print(f"Epoch [{epoch+1:3d}/{EPOCHS}] | Train Loss : {epoch_loss:.4f}", end="")
        
        if (epoch + 1) % EVAL_EVERY_N_EPOCHS == 0:
            model.eval()
            with torch.no_grad():
                test_outputs = model(Z_test_concat)
                _, predicted = torch.max(test_outputs.data, 1)
                acc = (predicted == y_test_tensor).sum().item() / y_test_tensor.size(0)
            print(f" | Test Acc : {acc:.4f}")
        else:
            print() 
            
model.eval()
with torch.no_grad():
    test_outputs = model(Z_test_concat)
    _, predicted = torch.max(test_outputs.data, 1)
    final_acc = (predicted == y_test_tensor).sum().item() / y_test_tensor.size(0)
    
df_results.loc[mode, "Test Accuracy"] = round(final_acc, 3)

Epoch [  1/1000] | Train Loss : 2.5732
Epoch [  5/1000] | Train Loss : 2.2269
Epoch [ 10/1000] | Train Loss : 2.0220 | Test Acc : 0.3277
Epoch [ 15/1000] | Train Loss : 1.9158
Epoch [ 20/1000] | Train Loss : 1.8305 | Test Acc : 0.4205
Epoch [ 25/1000] | Train Loss : 1.7574
Epoch [ 30/1000] | Train Loss : 1.7009 | Test Acc : 0.4615
Epoch [ 35/1000] | Train Loss : 1.6577
Epoch [ 40/1000] | Train Loss : 1.6227 | Test Acc : 0.4911
Epoch [ 45/1000] | Train Loss : 1.5930
Epoch [ 50/1000] | Train Loss : 1.5653 | Test Acc : 0.5085
Epoch [ 55/1000] | Train Loss : 1.5364
Epoch [ 60/1000] | Train Loss : 1.5186 | Test Acc : 0.5207
Epoch [ 65/1000] | Train Loss : 1.5015
Epoch [ 70/1000] | Train Loss : 1.4811 | Test Acc : 0.5304
Epoch [ 75/1000] | Train Loss : 1.4629
Epoch [ 80/1000] | Train Loss : 1.4538 | Test Acc : 0.5381
Epoch [ 85/1000] | Train Loss : 1.4439
Epoch [ 90/1000] | Train Loss : 1.4248 | Test Acc : 0.5430
Epoch [ 95/1000] | Train Loss : 1.4187
Epoch [100/1000] | Train Loss : 1.4060 |

In [ ]:
mode = "per_scale"

model = DynamicAttentionClassifier(
    num_vars=NUM_VARS, 
    num_classes=num_classes, 
    patches_counts=patches_counts, 
    mode=mode
).to(device)

_ = model(Z_train_concat[:2]) # Dummy pass for LazyLinear

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    
    for batch_z, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_z)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * batch_z.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    

    if (epoch + 1) == 1 or (epoch + 1) % PRINT_LOSS_EVERY == 0:
        print(f"Epoch [{epoch+1:3d}/{EPOCHS}] | Train Loss : {epoch_loss:.4f}", end="")
        
        if (epoch + 1) % EVAL_EVERY_N_EPOCHS == 0:
            model.eval()
            with torch.no_grad():
                test_outputs = model(Z_test_concat)
                _, predicted = torch.max(test_outputs.data, 1)
                acc = (predicted == y_test_tensor).sum().item() / y_test_tensor.size(0)
            print(f" | Test Acc : {acc:.4f}")
        else:
            print() 
            
model.eval()
with torch.no_grad():
    test_outputs = model(Z_test_concat)
    _, predicted = torch.max(test_outputs.data, 1)
    final_acc = (predicted == y_test_tensor).sum().item() / y_test_tensor.size(0)
    
df_results.loc[mode, "Test Accuracy"] = round(final_acc, 3)

Epoch [  1/1000] | Train Loss : 2.4546
Epoch [  5/1000] | Train Loss : 1.8087
Epoch [ 10/1000] | Train Loss : 1.6112 | Test Acc : 0.5024
Epoch [ 15/1000] | Train Loss : 1.4951
Epoch [ 20/1000] | Train Loss : 1.4152 | Test Acc : 0.5539
Epoch [ 25/1000] | Train Loss : 1.3630
Epoch [ 30/1000] | Train Loss : 1.3198 | Test Acc : 0.5746
Epoch [ 35/1000] | Train Loss : 1.2825
Epoch [ 40/1000] | Train Loss : 1.2541 | Test Acc : 0.5835
Epoch [ 45/1000] | Train Loss : 1.2284
Epoch [ 50/1000] | Train Loss : 1.2058 | Test Acc : 0.5908
Epoch [ 55/1000] | Train Loss : 1.1881
Epoch [ 60/1000] | Train Loss : 1.1654 | Test Acc : 0.5929
Epoch [ 65/1000] | Train Loss : 1.1436
Epoch [ 70/1000] | Train Loss : 1.1357 | Test Acc : 0.5957
Epoch [ 75/1000] | Train Loss : 1.1182
Epoch [ 80/1000] | Train Loss : 1.1063 | Test Acc : 0.5985
Epoch [ 85/1000] | Train Loss : 1.0902
Epoch [ 90/1000] | Train Loss : 1.0761 | Test Acc : 0.5969
Epoch [ 95/1000] | Train Loss : 1.0624
Epoch [100/1000] | Train Loss : 1.0518 |

In [ ]:
mode = "per_variable"

model = DynamicAttentionClassifier(
    num_vars=NUM_VARS, 
    num_classes=num_classes, 
    patches_counts=patches_counts, 
    mode=mode
).to(device)

_ = model(Z_train_concat[:2]) # Dummy pass for LazyLinear

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    
    for batch_z, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_z)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * batch_z.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    

    if (epoch + 1) == 1 or (epoch + 1) % PRINT_LOSS_EVERY == 0:
        print(f"Epoch [{epoch+1:3d}/{EPOCHS}] | Train Loss : {epoch_loss:.4f}", end="")
        
        if (epoch + 1) % EVAL_EVERY_N_EPOCHS == 0:
            model.eval()
            with torch.no_grad():
                test_outputs = model(Z_test_concat)
                _, predicted = torch.max(test_outputs.data, 1)
                acc = (predicted == y_test_tensor).sum().item() / y_test_tensor.size(0)
            print(f" | Test Acc : {acc:.4f}")
        else:
            print() 
            
model.eval()
with torch.no_grad():
    test_outputs = model(Z_test_concat)
    _, predicted = torch.max(test_outputs.data, 1)
    final_acc = (predicted == y_test_tensor).sum().item() / y_test_tensor.size(0)
    
df_results.loc[mode, "Test Accuracy"] = round(final_acc, 3)

Epoch [  1/1000] | Train Loss : 2.4016
Epoch [  5/1000] | Train Loss : 1.8149
Epoch [ 10/1000] | Train Loss : 1.5742 | Test Acc : 0.5134
Epoch [ 15/1000] | Train Loss : 1.4266
Epoch [ 20/1000] | Train Loss : 1.3243 | Test Acc : 0.5608
Epoch [ 25/1000] | Train Loss : 1.2464
Epoch [ 30/1000] | Train Loss : 1.1873 | Test Acc : 0.5864
Epoch [ 35/1000] | Train Loss : 1.1397
Epoch [ 40/1000] | Train Loss : 1.1004 | Test Acc : 0.5985
Epoch [ 45/1000] | Train Loss : 1.0611
Epoch [ 50/1000] | Train Loss : 1.0281 | Test Acc : 0.6099
Epoch [ 55/1000] | Train Loss : 0.9992
Epoch [ 60/1000] | Train Loss : 0.9723 | Test Acc : 0.6107
Epoch [ 65/1000] | Train Loss : 0.9433
Epoch [ 70/1000] | Train Loss : 0.9211 | Test Acc : 0.6119
Epoch [ 75/1000] | Train Loss : 0.9024
Epoch [ 80/1000] | Train Loss : 0.8755 | Test Acc : 0.6131
Epoch [ 85/1000] | Train Loss : 0.8564
Epoch [ 90/1000] | Train Loss : 0.8326 | Test Acc : 0.6156
Epoch [ 95/1000] | Train Loss : 0.8220
Epoch [100/1000] | Train Loss : 0.8044 |

In [ ]:
mode = "per_both"

model = DynamicAttentionClassifier(
    num_vars=NUM_VARS, 
    num_classes=num_classes, 
    patches_counts=patches_counts, 
    mode=mode
).to(device)

_ = model(Z_train_concat[:2]) # Dummy pass for LazyLinear

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    
    for batch_z, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_z)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * batch_z.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    

    if (epoch + 1) == 1 or (epoch + 1) % PRINT_LOSS_EVERY == 0:
        print(f"Epoch [{epoch+1:3d}/{EPOCHS}] | Train Loss : {epoch_loss:.4f}", end="")
        
        if (epoch + 1) % EVAL_EVERY_N_EPOCHS == 0:
            model.eval()
            with torch.no_grad():
                test_outputs = model(Z_test_concat)
                _, predicted = torch.max(test_outputs.data, 1)
                acc = (predicted == y_test_tensor).sum().item() / y_test_tensor.size(0)
            print(f" | Test Acc : {acc:.4f}")
        else:
            print() 
            
model.eval()
with torch.no_grad():
    test_outputs = model(Z_test_concat)
    _, predicted = torch.max(test_outputs.data, 1)
    final_acc = (predicted == y_test_tensor).sum().item() / y_test_tensor.size(0)
    
df_results.loc[mode, "Test Accuracy"] = round(final_acc, 3)

Epoch [  1/1000] | Train Loss : 2.0565
Epoch [  5/1000] | Train Loss : 1.2490
Epoch [ 10/1000] | Train Loss : 1.0207 | Test Acc : 0.6225
Epoch [ 15/1000] | Train Loss : 0.8901
Epoch [ 20/1000] | Train Loss : 0.7820 | Test Acc : 0.6403
Epoch [ 25/1000] | Train Loss : 0.7095
Epoch [ 30/1000] | Train Loss : 0.6394 | Test Acc : 0.6464
Epoch [ 35/1000] | Train Loss : 0.5824
Epoch [ 40/1000] | Train Loss : 0.5296 | Test Acc : 0.6387
Epoch [ 45/1000] | Train Loss : 0.4894
Epoch [ 50/1000] | Train Loss : 0.4488 | Test Acc : 0.6346
Epoch [ 55/1000] | Train Loss : 0.4175
Epoch [ 60/1000] | Train Loss : 0.3866 | Test Acc : 0.6387
Epoch [ 65/1000] | Train Loss : 0.3612
Epoch [ 70/1000] | Train Loss : 0.3359 | Test Acc : 0.6322
Epoch [ 75/1000] | Train Loss : 0.3099
Epoch [ 80/1000] | Train Loss : 0.2878 | Test Acc : 0.6302
Epoch [ 85/1000] | Train Loss : 0.2684
Epoch [ 90/1000] | Train Loss : 0.2525 | Test Acc : 0.6318
Epoch [ 95/1000] | Train Loss : 0.2386
Epoch [100/1000] | Train Loss : 0.2252 |

In [ ]:
display(df_results)

,Test Accuracy
Attention mode,
global,0.592
per_scale,0.555
per_variable,0.558
per_both,0.599


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# 1. Définition du nouveau modèle Multi-Head Attention
class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, in_features=384, num_heads=8, num_classes=2, dropout=0.01):
        super().__init__()
        # Token apprenable [CLS] (la "Query" qui va interroger les patchs)
        self.cls_token = nn.Parameter(torch.randn(1, 1, in_features) * 0.02)
        
        # La fameuse couche d'attention multi-têtes native de PyTorch
        self.mha = nn.MultiheadAttention(
            embed_dim=in_features, 
            num_heads=num_heads, 
            dropout=dropout, 
            batch_first=True
        )
        
        # Classifieur final (MLP un peu plus robuste qu'un simple Linear)
        self.lin = nn.Linear(in_features, num_classes)

    def forward(self, x):
        # x shape: (Batch, Seq_Len=66, Features=384)
        B = x.shape[0]
        
        # On duplique le token [CLS] pour l'avoir sur tout le batch
        cls_tokens = self.cls_token.expand(B, -1, -1)  # (Batch, 1, 384)
        
        # Cross-Attention : Le [CLS] (Query) regarde tous tes patchs (Key / Value)
        attn_output, attn_weights = self.mha(query=cls_tokens, key=x, value=x)
        
        # attn_output est de taille (Batch, 1, 384). On enlève la dimension 1 inutile avec squeeze
        pooled = attn_output.squeeze(1)  # (Batch, 384)
        
        # On passe l'embedding agrégé dans le classifieur
        return self.lin(pooled)


# 2. Paramètres d'entraînement (identiques aux tiens)
BATCH_SIZE = 64*2**2
EPOCHS = 1000
LEARNING_RATE = 0.0001
EVAL_EVERY_N_EPOCHS = 10
PRINT_LOSS_EVERY = 5 

# 3. Préparation du DataLoader
train_dataset = TensorDataset(Z_train_concat, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# 4. Instanciation du modèle et de l'optimiseur
model_mha = MultiHeadAttentionPooling(
    in_features=384, 
    num_heads=4, 
    num_classes=num_classes, # Utilise ta variable déjà existante
    dropout=0.01
).to(device)

optimizer = optim.Adam(model_mha.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

print("🚀 Début de l'entraînement avec Multi-Head Attention Pooling...")

# 5. Boucle d'entraînement
for epoch in range(EPOCHS):
    model_mha.train()
    running_loss = 0.0
    
    for batch_z, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model_mha(batch_z)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * batch_z.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    
    # Affichage des logs
    if (epoch + 1) == 1 or (epoch + 1) % PRINT_LOSS_EVERY == 0:
        print(f"Epoch [{epoch+1:3d}/{EPOCHS}] | Train Loss : {epoch_loss:.4f}", end="")
        
        # Évaluation périodique sur le test set
        if (epoch + 1) % EVAL_EVERY_N_EPOCHS == 0:
            model_mha.eval()
            with torch.no_grad():
                test_outputs = model_mha(Z_test_concat)
                _, predicted = torch.max(test_outputs.data, 1)
                acc = (predicted == y_test_tensor).sum().item() / y_test_tensor.size(0)
            print(f" | Test Acc : {acc:.4f}")
        else:
            print() 
            
# 6. Évaluation finale
model_mha.eval()
with torch.no_grad():
    test_outputs = model_mha(Z_test_concat)
    _, predicted = torch.max(test_outputs.data, 1)
    final_acc = (predicted == y_test_tensor).sum().item() / y_test_tensor.size(0)
    
print(f"\n🏆 Précision finale sur le Test Set : {final_acc:.3f}")

🚀 Début de l'entraînement avec Multi-Head Attention Pooling...
Epoch [  1/1000] | Train Loss : 2.4292
Epoch [  5/1000] | Train Loss : 2.0335
Epoch [ 10/1000] | Train Loss : 1.8379 | Test Acc : 0.4465
Epoch [ 15/1000] | Train Loss : 1.6199
Epoch [ 20/1000] | Train Loss : 1.4592 | Test Acc : 0.5251
Epoch [ 25/1000] | Train Loss : 1.3577
Epoch [ 30/1000] | Train Loss : 1.3010 | Test Acc : 0.5689
Epoch [ 35/1000] | Train Loss : 1.2585
Epoch [ 40/1000] | Train Loss : 1.2248 | Test Acc : 0.5892
Epoch [ 45/1000] | Train Loss : 1.1917
Epoch [ 50/1000] | Train Loss : 1.1547 | Test Acc : 0.5880
Epoch [ 55/1000] | Train Loss : 1.1254
Epoch [ 60/1000] | Train Loss : 1.0939 | Test Acc : 0.5900
Epoch [ 65/1000] | Train Loss : 1.0648
Epoch [ 70/1000] | Train Loss : 1.0368 | Test Acc : 0.5929
Epoch [ 75/1000] | Train Loss : 1.0036
Epoch [ 80/1000] | Train Loss : 0.9753 | Test Acc : 0.5872
Epoch [ 85/1000] | Train Loss : 0.9533
Epoch [ 90/1000] | Train Loss : 0.9230 | Test Acc : 0.5811
Epoch [ 95/1000]

In [ ]:
import torch
import torch.nn as nn

class ImprovedMultiHeadAttention(nn.Module):
    def __init__(self, in_features=384, seq_len=66, num_heads=8, num_classes=2, dropout=0.1):
        super().__init__()
        
        # 1. Le Manager (Token CLS)
        self.cls_token = nn.Parameter(torch.randn(1, 1, in_features) * 0.02)
        
        # 2. Le Positional Encoding (Le remède contre l'amnésie)
        self.pos_embed = nn.Parameter(torch.randn(1, seq_len, in_features) * 0.02)
        
        # 3. Layer Normalization (Pour la stabilité avant l'attention)
        self.norm_k = nn.LayerNorm(in_features)
        self.norm_q = nn.LayerNorm(in_features)
        
        # 4. L'Attention Multi-Têtes
        self.mha = nn.MultiheadAttention(
            embed_dim=in_features, 
            num_heads=num_heads, 
            dropout=dropout, 
            batch_first=True
        )
        
        # 5. La Tête de Classification : Juste Linéaire !
        self.classifier = nn.Linear(in_features, num_classes)

    def forward(self, x):
        # x shape: (Batch, Seq_Len=66, Features=384)
        B = x.shape[0]
        
        # On ajoute l'encodage de position aux patchs AVANT l'attention
        x = x + self.pos_embed
        
        # On duplique le token CLS pour le batch
        cls_tokens = self.cls_token.expand(B, -1, -1)
        
        # Normalisation pour stabiliser l'entraînement
        x_norm = self.norm_k(x)
        cls_norm = self.norm_q(cls_tokens)
        
        # Cross-Attention
        attn_output, _ = self.mha(query=cls_norm, key=x_norm, value=x_norm)
        
        # On retire la dimension de la séquence qui vaut 1
        pooled = attn_output.squeeze(1)
        
        # On passe directement dans la couche linéaire
        return self.classifier(pooled)

In [ ]:
model_mha = ImprovedMultiHeadAttention(
    in_features=384, 
    seq_len=66, # Tu as 66 patchs au total d'après ton print
    num_heads=8, 
    num_classes=num_classes, 
    dropout=0.1
).to(device)

optimizer = optim.Adam(model_mha.parameters(), lr=0.0001, weight_decay=1e-3)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd

# ==========================================
# 1. DÉFINITION DU MODÈLE
# ==========================================
class ImprovedMultiHeadAttention(nn.Module):
    def __init__(self, in_features=384, seq_len=66, num_heads=8, num_classes=2, dropout=0.1):
        super().__init__()
        
        # Le Manager (Token CLS)
        self.cls_token = nn.Parameter(torch.randn(1, 1, in_features) * 0.02)
        
        # Le Positional Encoding (Le remède contre l'amnésie)
        self.pos_embed = nn.Parameter(torch.randn(1, seq_len, in_features) * 0.02)
        
        # Layer Normalization (Pour la stabilité avant l'attention)
        self.norm_k = nn.LayerNorm(in_features)
        self.norm_q = nn.LayerNorm(in_features)
        
        # L'Attention Multi-Têtes
        self.mha = nn.MultiheadAttention(
            embed_dim=in_features, 
            num_heads=num_heads, 
            dropout=dropout, 
            batch_first=True
        )
        
        # La Tête de Classification : Juste Linéaire !
        self.classifier = nn.Linear(in_features, num_classes)

    def forward(self, x):
        B = x.shape[0]
        
        # On ajoute l'encodage de position aux patchs AVANT l'attention
        x = x + self.pos_embed
        
        # On duplique le token CLS pour le batch
        cls_tokens = self.cls_token.expand(B, -1, -1)
        
        # Normalisation pour stabiliser l'entraînement
        x_norm = self.norm_k(x)
        cls_norm = self.norm_q(cls_tokens)
        
        # Cross-Attention
        attn_output, _ = self.mha(query=cls_norm, key=x_norm, value=x_norm)
        
        # On retire la dimension de la séquence
        pooled = attn_output.squeeze(1)
        
        # Prédiction finale
        return self.classifier(pooled)


# ==========================================
# 2. PARAMÈTRES ET PRÉPARATION
# ==========================================
BATCH_SIZE = 64*2**2
EPOCHS = 1000
LEARNING_RATE = 0.00001
EVAL_EVERY_N_EPOCHS = 10
PRINT_LOSS_EVERY = 5 
WEIGHT_DECAY = 1e-3 # Plus élevé pour lutter contre l'overfitting

# On s'assure d'avoir notre DataFrame pour stocker les résultats
if 'df_results' not in locals():
    df_results = pd.DataFrame(columns=["Test Accuracy"])
    df_results.index.name = "Attention mode"

mode_name = "multi_head_cls_linear"

# Préparation du DataLoader
train_dataset = TensorDataset(Z_train_concat, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Instanciation du modèle
# seq_len=66 correspond au nombre total de patchs concaténés (5+3+2+1 * 6 variables)
model = ImprovedMultiHeadAttention(
    in_features=384, 
    seq_len=66, 
    num_heads=16, 
    num_classes=num_classes, 
    dropout=0.0
).to(device)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss()

# ==========================================
# 3. BOUCLE D'ENTRAÎNEMENT
# ==========================================
print(f"🚀 Début de l'entraînement pour le mode : {mode_name}")

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    
    for batch_z, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_z)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * batch_z.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    
    # Affichage régulier
    if (epoch + 1) == 1 or (epoch + 1) % PRINT_LOSS_EVERY == 0:
        print(f"Epoch [{epoch+1:3d}/{EPOCHS}] | Train Loss : {epoch_loss:.4f}", end="")
        
        if (epoch + 1) % EVAL_EVERY_N_EPOCHS == 0:
            model.eval()
            with torch.no_grad():
                test_outputs = model(Z_test_concat)
                _, predicted = torch.max(test_outputs.data, 1)
                acc = (predicted == y_test_tensor).sum().item() / y_test_tensor.size(0)
            print(f" | Test Acc : {acc:.4f}")
        else:
            print() 

# ==========================================
# 4. ÉVALUATION FINALE ET SAUVEGARDE
# ==========================================
model.eval()
with torch.no_grad():
    test_outputs = model(Z_test_concat)
    _, predicted = torch.max(test_outputs.data, 1)
    final_acc = (predicted == y_test_tensor).sum().item() / y_test_tensor.size(0)

# Enregistrement dans le DataFrame
df_results.loc[mode_name, "Test Accuracy"] = round(final_acc, 3)

print("\n" + "="*40)
print(f"🏆 Précision finale sur le Test Set : {final_acc:.3f}")
print("="*40)

# Afficher le tableau des résultats mis à jour
display(df_results)

🚀 Début de l'entraînement pour le mode : multi_head_cls_linear
Epoch [  1/1000] | Train Loss : 2.5486


Epoch [  5/1000] | Train Loss : 2.3399
Epoch [ 10/1000] | Train Loss : 2.1800 | Test Acc : 0.3151
Epoch [ 15/1000] | Train Loss : 2.0910
Epoch [ 20/1000] | Train Loss : 2.0277 | Test Acc : 0.3402
Epoch [ 25/1000] | Train Loss : 1.9690
Epoch [ 30/1000] | Train Loss : 1.9100 | Test Acc : 0.4063
Epoch [ 35/1000] | Train Loss : 1.8511
Epoch [ 40/1000] | Train Loss : 1.7947 | Test Acc : 0.4331
Epoch [ 45/1000] | Train Loss : 1.7417
Epoch [ 50/1000] | Train Loss : 1.6922 | Test Acc : 0.4651
Epoch [ 55/1000] | Train Loss : 1.6461
Epoch [ 60/1000] | Train Loss : 1.6030 | Test Acc : 0.4846
Epoch [ 65/1000] | Train Loss : 1.5628
Epoch [ 70/1000] | Train Loss : 1.5260 | Test Acc : 0.5126
Epoch [ 75/1000] | Train Loss : 1.4924
Epoch [ 80/1000] | Train Loss : 1.4616 | Test Acc : 0.5324
Epoch [ 85/1000] | Train Loss : 1.4337
Epoch [ 90/1000] | Train Loss : 1.4084 | Test Acc : 0.5466
Epoch [ 95/1000] | Train Loss : 1.3853
Epoch [100/1000] | Train Loss : 1.3641 | Test Acc : 0.5608
Epoch [105/1000] | T

KeyboardInterrupt: 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd

# ==========================================
# 1. DÉFINITION DU MODÈLE
# ==========================================
class PerScaleVariableAttentionFlatten(nn.Module):
    def __init__(self, in_features=384, num_vars=6, patches_counts=[5, 3, 2, 1], num_heads=4, num_classes=2, dropout=0.1):
        super().__init__()
        self.num_vars = num_vars
        self.patches_counts = patches_counts
        self.num_scales = len(patches_counts)
        self.in_features = in_features
        
        # On crée une liste de paramètres apprenables (1 pour chaque échelle)
        # - Un token CLS par échelle
        # - Un Positional Encoding par échelle (taille = nb de patches pour cette échelle)
        self.cls_tokens = nn.ParameterList([
            nn.Parameter(torch.randn(1, 1, in_features) * 0.02) for _ in range(self.num_scales)
        ])
        
        self.pos_embeds = nn.ParameterList([
            nn.Parameter(torch.randn(1, p_count, in_features) * 0.02) for p_count in patches_counts
        ])
        
        # On crée 4 couches d'attention indépendantes (1 par échelle)
        self.mhas = nn.ModuleList([
            nn.MultiheadAttention(embed_dim=in_features, num_heads=num_heads, dropout=dropout, batch_first=True)
            for _ in range(self.num_scales)
        ])
        
        # Classifieur linéaire final
        # Il va recevoir (6 variables) x (4 échelles) = 24 résumés de taille 384
        # Donc 24 * 384 = 9216 features en entrée
        flat_dim = self.num_vars * self.num_scales * in_features
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(flat_dim, num_classes)
        )

    def forward(self, x):
            # x shape: (Batch, Total_Patches=66, Features=384)
            B = x.shape[0]
            
            # 1. On redécoupe le tenseur global en ses 4 échelles
            split_sizes = [p * self.num_vars for p in self.patches_counts] # ex: [30, 18, 12, 6]
            x_splits = torch.split(x, split_sizes, dim=1)
            
            all_pooled = []
            
            # 2. On boucle sur chaque échelle
            for i, p_count in enumerate(self.patches_counts):
                x_i = x_splits[i]  # shape: (Batch, p_count * 6, 384)
                
                # On sépare les variables avec RESHAPE (au lieu de view)
                x_i = x_i.reshape(B, self.num_vars, p_count, self.in_features)
                
                # On fusionne Batch et Variables avec RESHAPE
                x_fused = x_i.reshape(B * self.num_vars, p_count, self.in_features)
                
                # On ajoute le Positional Encoding spécifique à cette échelle
                x_fused = x_fused + self.pos_embeds[i]
                
                # On prépare le CLS token et on l'étend
                cls_expanded = self.cls_tokens[i].expand(B * self.num_vars, -1, -1)
                
                # Multi-Head Attention
                attn_out, _ = self.mhas[i](query=cls_expanded, key=x_fused, value=x_fused)
                
                # On enlève la dimension de séquence
                pooled = attn_out.squeeze(1)
                
                # On sépare à nouveau Batch et Variables et on "Flatten" avec RESHAPE
                pooled_flat = pooled.reshape(B, self.num_vars * self.in_features)
                
                all_pooled.append(pooled_flat)
                
            # 3. Flatten final (On concatène les 4 échelles)
            final_flat = torch.cat(all_pooled, dim=1)
            
            # 4. Prédiction linéaire
            return self.classifier(final_flat)


# ==========================================
# 2. PARAMÈTRES ET PRÉPARATION
# ==========================================
BATCH_SIZE = 64*2**2
EPOCHS = 1000
LEARNING_RATE = 0.00001
EVAL_EVERY_N_EPOCHS = 10
PRINT_LOSS_EVERY = 5 
WEIGHT_DECAY = 1e-3

if 'df_results' not in locals():
    df_results = pd.DataFrame(columns=["Test Accuracy"])
    df_results.index.name = "Attention mode"

mode_name = "attention_per_scale_var_flatten"

# DataLoader
train_dataset = TensorDataset(Z_train_concat, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Instanciation
model = PerScaleVariableAttentionFlatten(
    in_features=384, 
    num_vars=NUM_VARS, # 6
    patches_counts=patches_counts, # [5, 3, 2, 1]
    num_heads=4, 
    num_classes=num_classes, 
    dropout=0.1
).to(device)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss()

# ==========================================
# 3. BOUCLE D'ENTRAÎNEMENT
# ==========================================
print(f"🚀 Début de l'entraînement pour le mode : {mode_name}")

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    
    for batch_z, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_z)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * batch_z.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    
    if (epoch + 1) == 1 or (epoch + 1) % PRINT_LOSS_EVERY == 0:
        print(f"Epoch [{epoch+1:3d}/{EPOCHS}] | Train Loss : {epoch_loss:.4f}", end="")
        
        if (epoch + 1) % EVAL_EVERY_N_EPOCHS == 0:
            model.eval()
            with torch.no_grad():
                test_outputs = model(Z_test_concat)
                _, predicted = torch.max(test_outputs.data, 1)
                acc = (predicted == y_test_tensor).sum().item() / y_test_tensor.size(0)
            print(f" | Test Acc : {acc:.4f}")
        else:
            print() 

# ==========================================
# 4. ÉVALUATION FINALE ET SAUVEGARDE
# ==========================================
model.eval()
with torch.no_grad():
    test_outputs = model(Z_test_concat)
    _, predicted = torch.max(test_outputs.data, 1)
    final_acc = (predicted == y_test_tensor).sum().item() / y_test_tensor.size(0)

df_results.loc[mode_name, "Test Accuracy"] = round(final_acc, 3)

print("\n" + "="*40)
print(f"🏆 Précision finale sur le Test Set : {final_acc:.3f}")
print("="*40)

display(df_results)

🚀 Début de l'entraînement pour le mode : attention_per_scale_var_flatten
Epoch [  1/1000] | Train Loss : 2.6025
Epoch [  5/1000] | Train Loss : 2.1316
Epoch [ 10/1000] | Train Loss : 1.9188 | Test Acc : 0.3982
Epoch [ 15/1000] | Train Loss : 1.7961
Epoch [ 20/1000] | Train Loss : 1.6944 | Test Acc : 0.4619
Epoch [ 25/1000] | Train Loss : 1.6112
Epoch [ 30/1000] | Train Loss : 1.5413 | Test Acc : 0.5109
Epoch [ 35/1000] | Train Loss : 1.4774
Epoch [ 40/1000] | Train Loss : 1.4211 | Test Acc : 0.5377
Epoch [ 45/1000] | Train Loss : 1.3729
Epoch [ 50/1000] | Train Loss : 1.3335 | Test Acc : 0.5547
Epoch [ 55/1000] | Train Loss : 1.2974
Epoch [ 60/1000] | Train Loss : 1.2653 | Test Acc : 0.5750
Epoch [ 65/1000] | Train Loss : 1.2401
Epoch [ 70/1000] | Train Loss : 1.2110 | Test Acc : 0.5880
Epoch [ 75/1000] | Train Loss : 1.1876
Epoch [ 80/1000] | Train Loss : 1.1676 | Test Acc : 0.5945
Epoch [ 85/1000] | Train Loss : 1.1472
Epoch [ 90/1000] | Train Loss : 1.1290 | Test Acc : 0.6010
Epoch 

KeyboardInterrupt: 

TODO :
- target_dim?
- result diff de maxime car moi moi pytorch pour reglog